In [ ]:
from __future__ import annotations

import argparse, csv, os, pathlib, re, sys, time
from dataclasses import dataclass
from typing import Iterator, List, Tuple

import pandas as pd
from bs4 import BeautifulSoup  # pip install beautifulsoup4
from selenium import webdriver  # pip install selenium~=4.21.0
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib.parse

###############################################################################
# 0. Config & helpers
###############################################################################

BLOCKWORDS = {
    "협찬", "체험단", "서포터즈", "공구", "원고료", "지원받아", "스폰", "서평", "AD", "ad",
    "광고", "홍보", "리뷰어", "샘플", "provided", "sponsored", "gifted",
}

# CSS 선택자 (Selectors)
BLOG_LINK_SELECTOR = "a.desc_inner"
PUBLISH_DATE_SEL   = "span.se_publishDate, span.date"
CONTENT_SEL        = "div.se-main-container, #postViewArea"

try:
    SCRIPT_DIR = pathlib.Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = pathlib.Path.cwd()


def sanitize(text: str) -> str:
    """공백을 하나로 줄이고 제어 문자를 제거하여 텍스트를 정리합니다."""
    clean = re.sub(r"[\x00-\x1f\u2028\u2029]+", " ", text)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean


def is_sponsored(text: str, allow: bool = False) -> bool:
    """텍스트에 협찬/광고 관련 단어가 포함되어 있는지 확인합니다."""
    if allow:
        return False
    lower = text.lower()
    return any(word.lower() in lower for word in BLOCKWORDS)


###############################################################################
# 1. Selenium boilerplate
###############################################################################

def make_driver(headless: bool = True) -> webdriver.Chrome:
    """Selenium Chrome 웹 드라이버 인스턴스를 생성합니다."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--window-size=1280,1024")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


###############################################################################
# 2. Core crawler
###############################################################################

@dataclass
class Review:
    """수집된 리뷰 데이터를 저장하기 위한 데이터 클래스입니다."""
    search_keyword: str
    place_name: str
    address: str
    url: str
    date: str
    text: str

    def as_tuple(self) -> Tuple[str, str, str, str, str, str]:
        """데이터를 CSV 저장을 위해 튜플 형태로 변환합니다."""
        return (
            self.search_keyword,
            self.place_name,
            self.address,
            self.url,
            self.date,
            self.text,
        )

# ################## 이 함수를 아래 코드로 교체해주세요 ##################
def search_blog_links(driver: webdriver.Chrome, keyword: str, max_links: int) -> List[str]:
    """네이버 블로그 검색을 URL로 직접 요청하고 링크를 수집합니다."""
    links: list[str] = []
    print(f"   - '{keyword}' 검색 시작...")

    # 1. 검색어를 URL 인코딩하여 검색 결과 페이지로 바로 이동
    encoded_keyword = urllib.parse.quote(keyword)
    
    # 페이지 번호를 바꿔가며 URL을 직접 호출할 것이므로 초기 페이지는 1로 설정
    current_page = 1
    
    while True:
        # 페이지 번호를 포함한 검색 URL 생성
        search_url = f"https://section.blog.naver.com/Search/Post.naver?pageNo={current_page}&rangeType=ALL&orderBy=sim&keyword={encoded_keyword}"
        
        try:
            driver.get(search_url)
            # 페이지가 로드될 시간을 줍니다.
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.area_list_search"))
            )
        except Exception as e:
            print(f"[경고] 검색 결과 페이지({search_url}) 로딩에 실패했습니다: {e}", file=sys.stderr)
            break

        # 2. 현재 페이지의 모든 블로그 링크 수집
        elements = driver.find_elements(By.CSS_SELECTOR, BLOG_LINK_SELECTOR)
        if not elements:
            print("[정보] 현재 페이지에서 더 이상 링크를 찾지 못해 중단합니다.")
            break

        new_links_found_on_page = False
        for a in elements:
            url = a.get_attribute("href")
            if url and url not in links:
                links.append(url)
                new_links_found_on_page = True

        print(f"     - {current_page} 페이지에서 {len(elements)}개 링크 확인, 총 {len(links)}개 수집됨.")

        # 3. 목표 수량 도달 시 중단
        if len(links) >= max_links:
            print(f"[정보] 목표 수량({max_links}개) 이상의 링크를 탐색하여 중단합니다.")
            break
        
        # 4. 다음 페이지로 이동
        try:
            # 현재 페이지네이션 블록에서 다음 번호로 이동
            pagination = driver.find_element(By.CLASS_NAME, "pagination")
            current_page_element = pagination.find_element(By.CSS_SELECTOR, "strong.current")
            
            # 다음 페이지 번호(<a> 태그)가 있는지 확인
            next_page_link = current_page_element.find_element(By.XPATH, "./parent::span/following-sibling::span/a")
            current_page += 1
            
        except NoSuchElementException:
            # 페이지 번호가 더 없다면 '다음' 버튼으로 페이지 블록을 넘김
            try:
                next_block_button = driver.find_element(By.CSS_SELECTOR, "a.button_next")
                # '다음' 버튼이 비활성화(disabled) 상태가 아니어야 함
                if "disabled" in next_block_button.get_attribute("class"):
                    print("[정보] 마지막 페이지 블록에 도달하여 수집을 중단합니다.")
                    break
                current_page += 1
            except NoSuchElementException:
                print("[정보] 모든 페이지를 확인하여 수집을 중단합니다.")
                break

    return links[:max_links]

def extract_post(driver: webdriver.Chrome, url: str) -> tuple[str | None, str | None]:
    """개별 블로그 포스트 URL에 접속하여 날짜와 본문 텍스트를 추출합니다."""
    try:
        driver.get(url)
        time.sleep(1.4)
        if "blog.naver.com" in driver.current_url and "PostView.naver" not in driver.current_url:
            try:
                driver.switch_to.frame("mainFrame")
            except Exception:
                pass

        soup = BeautifulSoup(driver.page_source, "html.parser")
        date_elt = soup.select_one(PUBLISH_DATE_SEL)
        date = sanitize(date_elt.get_text()) if date_elt else ""
        content_elt = soup.select_one(CONTENT_SEL)
        text = sanitize(content_elt.get_text(separator=" ")) if content_elt else ""
        driver.switch_to.default_content()
        return date, text
    except Exception as e:
        print(f"[경고] 포스트 추출 중 오류 발생: {e} – {url}", file=sys.stderr)
        return None, None


def crawl_reviews(
    keyword: str,
    place_name: str,
    address: str,
    *,
    search_limit: int,
    allow_sponsored: bool = False,
    driver: webdriver.Chrome | None = None,
    done_urls: set[str] | None = None,
) -> Iterator[Review]:
    """주어진 키워드로 리뷰를 크롤링합니다."""
    close_driver = False
    if driver is None:
        driver = make_driver(headless=True)
        close_driver = True
    
    if done_urls is None:
        done_urls = set()

    try:
        links = search_blog_links(driver, keyword, max_links=search_limit)
        print(f"   - 총 {len(links)}개 링크 수집 (최대 {search_limit}개까지 확인)")

        for i, link in enumerate(links):
            if link in done_urls:
                continue

            print(f"     - 포스트 {i+1}/{len(links)} 추출 중...")
            date, text = extract_post(driver, link)
            if not (date and text):
                continue
            if is_sponsored(text, allow=allow_sponsored):
                print(f"       - 광고/협찬 포스트 제외: {link}")
                continue
            
            yield Review(keyword, place_name, address, link, date, text)
    finally:
        if close_driver:
            driver.quit()


###############################################################################
# 3. CLI batch pipeline
###############################################################################

def open_csv_writer(path: pathlib.Path, *, resume: bool) -> tuple[csv.writer, any]:
    mode = "a" if resume and path.exists() else "w"
    f = open(path, mode, newline="", encoding="utf-8-sig")
    writer = csv.writer(f)
    if mode == "w" or os.stat(path).st_size == 0:
        writer.writerow(["search_keyword", "place_name", "address", "url", "date", "review_text"])
        f.flush()
    return writer, f


def load_done_info(path: pathlib.Path) -> tuple[dict[str, int], set[str]]:
    """CSV에서 place_name별 리뷰 개수와 저장된 URL 세트를 반환합니다."""
    if not path.exists() or os.stat(path).st_size == 0:
        return {}, set()
    try:
        df = pd.read_csv(path, usecols=["place_name", "url"])
        counts = df["place_name"].value_counts().to_dict()
        urls = set(df["url"].dropna())
        return counts, urls
    except (ValueError, KeyError) as e:
        print(f"[경고] CSV 파일('{path}')을 읽는 중 오류: {e}", file=sys.stderr)
        print("[경고] 'place_name'과 'url' 컬럼이 있는지 확인하세요. 이어쓰기를 비활성화합니다.", file=sys.stderr)
        return {}, set()


# ################## 이 함수가 수정되었습니다 ##################
def run_batch(
    input_df: pd.DataFrame,
    output_csv: pathlib.Path,
    *,
    max_posts: int = 50,       # 한 장소당 목표 리뷰 수
    skip_threshold: int = 15,  # 이미 이 개수 이상 수집된 곳은 스킵
    search_limit: int = 200,   # 한 장소당 최대 뒤질 링크 수
    rate_limit: float = 1.0,
    allow_sponsored: bool = False,
    resume: bool = True,
):
    """
    - 각 행(place)마다:
      1) CSV에 이미 모아둔 리뷰 개수(load_done_info) 로 resume 지원
      2) 이미 skip_threshold 이상이면 건너뛰고
      3) max_posts - already 만큼(최대 50개) 크롤링, 광고/협찬 제외 후 카운트
      4) 200개 링크 후보 중에서만 검색
    """
    # 1) 기존 데이터 로드
    done_counts, done_urls = load_done_info(output_csv) if resume else ({}, set())
    local_counts = dict(done_counts)

    # 2) CSV writer, driver 준비
    writer, fh = open_csv_writer(output_csv, resume=resume)
    driver = make_driver(headless=True)

    df = input_df.copy()
    df['address'] = df['addr1'].fillna('') + ' ' + df['addr2'].fillna('')

    try:
        for idx, row in df.iterrows():
            place = row['title']
            already = local_counts.get(place, 0)

            # 1) 충분히 모아뒀으면 스킵
            if already >= skip_threshold:
                print(f"[{idx+1}/{len(df)}] SKIP '{place}' (already={already} ≥ skip_threshold)")
                continue

            # 2) 남은 수 계산
            needed = max_posts - already
            if needed <= 0:
                print(f"[{idx+1}/{len(df)}] DONE '{place}' (already={already} ≥ max_posts)")
                continue

            print(f"[{idx+1}/{len(df)}] Start '{place}': need {needed} more reviews (already={already})")

            got = 0
            # 3) 후보 링크 고정 수 만큼 수집
            for rev in crawl_reviews(
                keyword=place,
                place_name=place,
                address=sanitize(row['address']),
                search_limit=search_limit,      # 여기서 200으로 충분히 뒤짐
                allow_sponsored=allow_sponsored,
                driver=driver,
                done_urls=done_urls,
            ):
                # 광고/협찬 포스트는 crawl_reviews 내부에서 걸러짐
                writer.writerow(rev.as_tuple()); fh.flush()
                done_urls.add(rev.url)

                got += 1
                local_counts[place] = already + got

                if got >= needed:
                    break

            # 4) 리뷰 하나도 못 모았으면 기록
            if got == 0 and already == 0:
                writer.writerow([place, place, sanitize(row['address']), "(no review)", "", ""])
                fh.flush()

            print(f"   → '{place}' collected {got} reviews (total now {local_counts[place]}/{max_posts})")
            time.sleep(rate_limit)

    finally:
        driver.quit()
        fh.close()
        print(f"✅ 작업 완료 — 데이터 저장 위치: {output_csv}")


###############################################################################
# ################## 이 부분이 수정되었습니다 ##################
if __name__ == "__main__":
    # --- 파일 이름 설정 ---
    input_excel_file = '전국_시도별/울산광역시.xlsx'
    output_csv_file = '전국_그룹분리_csv/울산광역시6.csv'

    print("크롤링을 시작합니다...")
    print(f"입력 파일: {input_excel_file}")
    print(f"출력 파일: {output_csv_file}")
    
    # 1. 전체 엑셀 파일을 읽어옵니다.
    try:
        full_df = pd.read_excel(input_excel_file)
    except FileNotFoundError:
        print(f"[오류] 입력 엑셀 파일 '{input_excel_file}'을 찾을 수 없습니다. 경로를 확인해주세요.")
        sys.exit(1)

    # 파이썬은 0부터 숫자를 셉니다. (엑셀 1행 = 0, 엑셀 2행 = 1)
    # 예시 1: 5번 행부터 10번 행까지 (엑셀 기준 6행 ~ 11행)
    start_row = 399
    end_row = 500
        # end_row가 None인 경우, 끝까지 슬라이싱합니다.
    end_slice = end_row + 1 if end_row is not None else None
    selected_df = full_df.iloc[start_row:end_slice]

    print(f"INFO: 총 {len(full_df)}개 중 선택된 {len(selected_df)}개 행에 대해 크롤링을 진행합니다.")

    # 3. 선택된 데이터만 run_batch 함수에 전달합니다.
    run_batch(
        input_df        = selected_df,
        output_csv      = pathlib.Path(output_csv_file),
        max_posts       = 50,    # 장소당 최대 50개 리뷰
        skip_threshold  = 15,    # 이미 15개 이상 있으면 스킵
        search_limit    = 200,   # 최대 200개 링크 뒤져서
        rate_limit      = 1.0,   # 페이지당 1초 대기
        allow_sponsored = False, # 광고/협찬 제외
        resume          = True,  # 기존 CSV 이어쓰기
    )

크롤링을 시작합니다...
입력 파일: 전국_시도별/울산광역시.xlsx
출력 파일: 전국_그룹분리_csv/울산광역시6.csv
INFO: 총 624개 중 선택된 102개 행에 대해 크롤링을 진행합니다.
[400/102] SKIP '울산큰애기상점가' (already=48 ≥ skip_threshold)
[401/102] SKIP '울산큰애기집' (already=48 ≥ skip_threshold)
[402/102] SKIP '울산큰애기청년야시장' (already=50 ≥ skip_threshold)
[403/102] SKIP '울산탁주 태화루' (already=50 ≥ skip_threshold)
[404/102] SKIP '울산테마식물수목원' (already=50 ≥ skip_threshold)
[405/102] SKIP '울산함' (already=50 ≥ skip_threshold)
[406/102] SKIP '울산해양박물관' (already=50 ≥ skip_threshold)
[407/102] SKIP '울산향교' (already=50 ≥ skip_threshold)
[408/102] SKIP '울주 간월사지 석조여래좌상' (already=50 ≥ skip_threshold)
[409/102] SKIP '울주 간절곶 해맞이 축제' (already=50 ≥ skip_threshold)
[410/102] SKIP '울주 구량리 은행나무' (already=50 ≥ skip_threshold)
[411/102] SKIP '울주 대곡리 반구대 암각화' (already=50 ≥ skip_threshold)
[412/102] SKIP '울주 드론 페스티벌' (already=50 ≥ skip_threshold)
[413/102] SKIP '울주 천전리 명문과 암각화' (already=48 ≥ skip_threshold)
[414/102] SKIP '울주 트레일 나인피크 대회' (already=50 ≥ skip_threshold)
[415/102] SKIP '울주문화예술회관